# Research-field and PhD-institution comparisons

This standalone notebook reads the existing standardized faculty outputs. It does not rerun or modify the preprocessing workflow.

Outputs:

- Complete group-comparison charts and CSV tables under `analysis/descriptive-stats-after-preprocessing/group_comparison/`.
- An integrated report at `docs/research-field-and-phd-institution-report.md`.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "data-preprocessing").exists():
            return candidate
    raise FileNotFoundError("Could not locate data/data-preprocessing from the current directory")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "data-preprocessing"
ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "descriptive-stats-after-preprocessing"
GROUP_COMPARISON_DIR = ANALYSIS_DIR / "group_comparison"
RESEARCH_OUTPUT_DIR = GROUP_COMPARISON_DIR / "research_fields"
PHD_OUTPUT_DIR = GROUP_COMPARISON_DIR / "phd_institutions"
DOCS_DIR = PROJECT_ROOT / "docs"
REPORT_OUTPUT = DOCS_DIR / "research-field-and-phd-institution-report.md"
for directory in [RESEARCH_OUTPUT_DIR, PHD_OUTPUT_DIR, DOCS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

GROUPS = {
    "fin-and-econ": ["nccu-fin", "nccu-mb", "nthu-qf", "ntu-econ", "ntu-fin", "nycu-info-mag-fin"],
    "ds": ["ncku-stats-ds", "ntu-comm-engin-ds-network", "ntu-ds"],
    "cs": ["ncku-cs-ie", "nthu-cs", "ntu-cs-ie", "ntu-im", "nycu-cs"],
}
RUN_TIMESTAMP = datetime.now().astimezone().isoformat(timespec="seconds")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 180, "font.size": 9})

print(f"Project root: {PROJECT_ROOT}")
print(f"Runtime: {sys.executable} (Python {sys.version.split()[0]})")

Project root: <project-root>
Runtime: python (Python 3.12.3)


In [2]:
research_long = pd.read_csv(DATA_DIR / "research_interests_standardized_long.csv")
standardized_records = json.loads((DATA_DIR / "faculty_standardized.json").read_text(encoding="utf-8"))
coverage_by_school = pd.read_csv(ANALYSIS_DIR / "coverage_by_school.csv")
top_research = pd.read_csv(ANALYSIS_DIR / "top_research_fields.csv")
top_phd = pd.read_csv(ANALYSIS_DIR / "top_phd_institutions.csv")

faculty_df = pd.DataFrame([
    {"faculty_id": record["faculty_id"], "school_id": record["school_id"]}
    for record in standardized_records
])
phd_rows = []
for record in standardized_records:
    for item in record.get("education_structured", []):
        if item.get("degree") == "PhD" and item.get("institution"):
            phd_rows.append({
                "faculty_id": record["faculty_id"],
                "school_id": record["school_id"],
                "institution_en": item["institution"],
            })
phd_long = (
    pd.DataFrame(phd_rows).drop_duplicates()
    if phd_rows
    else pd.DataFrame(columns=["faculty_id", "school_id", "institution_en"])
)

school_order = coverage_by_school["school_id"].tolist()
known_schools = set(school_order)
configured_schools = {school_id for members in GROUPS.values() for school_id in members}
missing_schools = sorted(configured_schools - known_schools)
assert not missing_schools, f"Unknown school IDs in GROUPS: {missing_schools}"
assert configured_schools == known_schools, f"GROUPS must cover every school exactly once; ungrouped: {sorted(known_schools - configured_schools)}"

print(f"Loaded {len(faculty_df):,} faculty, {len(research_long):,} research-field rows, and {len(phd_long):,} PhD-institution rows.")

Loaded 741 faculty, 1,953 research-field rows, and 482 PhD-institution rows.


## Generate complete group comparisons

Each heatmap and its companion CSV include every category observed in the group; no Top-N truncation is applied.

In [3]:
def group_comparison_table(long_df, value_col, group_name, school_ids):
    view = long_df[long_df["school_id"].isin(school_ids)].drop_duplicates(
        ["faculty_id", "school_id", value_col]
    )
    counts = (
        view.groupby([value_col, "school_id"])["faculty_id"]
        .nunique()
        .rename("faculty_count")
        .reset_index()
    )
    if counts.empty:
        return pd.DataFrame(columns=["group", value_col, *school_ids, "total_faculty_mentions", "department_count"])
    pivot = counts.pivot(index=value_col, columns="school_id", values="faculty_count").fillna(0).astype(int)
    pivot = pivot.reindex(columns=school_ids, fill_value=0)
    pivot["total_faculty_mentions"] = pivot.sum(axis=1)
    pivot["department_count"] = (pivot[school_ids] > 0).sum(axis=1)
    pivot = (
        pivot.reset_index()
        .sort_values(["total_faculty_mentions", "department_count", value_col], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    pivot.insert(0, "group", group_name)
    return pivot


def save_group_heatmap(table, value_col, group_name, school_ids, subject, output_dir, cmap):
    csv_path = output_dir / f"{group_name}.csv"
    png_path = output_dir / f"{group_name}.png"
    table.to_csv(csv_path, index=False, encoding="utf-8-sig")

    categories = table[value_col].tolist()
    matrix = table[school_ids].to_numpy(dtype=int) if categories else np.zeros((0, len(school_ids)), dtype=int)
    width = max(9, 1.55 * len(school_ids) + 4)
    height = max(5, 0.34 * max(len(categories), 6) + 2.8)
    fig, ax = plt.subplots(figsize=(width, height))
    if not categories:
        ax.axis("off")
        ax.text(0.5, 0.5, f"No standardized {subject.lower()} available", ha="center", va="center", fontsize=12)
    else:
        image = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0)
        ax.set_xticks(range(len(school_ids)), labels=school_ids, rotation=35, ha="right")
        ax.set_yticks(range(len(categories)), labels=categories)
        ax.set_xlabel("Program ID")
        ax.set_ylabel(subject)
        for row in range(matrix.shape[0]):
            for col in range(matrix.shape[1]):
                if matrix[row, col]:
                    ax.text(col, row, str(matrix[row, col]), ha="center", va="center", fontsize=7)
        fig.colorbar(image, ax=ax, label="Faculty count", shrink=0.7)
    ax.set_title(f"{group_name}: All {subject} by program")
    fig.tight_layout()
    fig.savefig(png_path, bbox_inches="tight")
    plt.close(fig)
    return [csv_path, png_path]


research_group_tables = {}
phd_group_tables = {}
group_output_files = []
for group_name, school_ids in GROUPS.items():
    research_table = group_comparison_table(research_long, "research_interest_en", group_name, school_ids)
    phd_table = group_comparison_table(phd_long, "institution_en", group_name, school_ids)
    research_group_tables[group_name] = research_table
    phd_group_tables[group_name] = phd_table
    group_output_files.extend(save_group_heatmap(
        research_table, "research_interest_en", group_name, school_ids,
        "Research Fields", RESEARCH_OUTPUT_DIR, "YlGnBu"
    ))
    group_output_files.extend(save_group_heatmap(
        phd_table, "institution_en", group_name, school_ids,
        "PhD Institutions", PHD_OUTPUT_DIR, "YlOrBr"
    ))

print(f"Generated {len(group_output_files)} group-comparison files under {GROUP_COMPARISON_DIR.relative_to(PROJECT_ROOT)}")

Generated 12 group-comparison files under analysis\descriptive-stats-after-preprocessing\group_comparison


## Build the integrated analysis and comparison report

In [4]:
def value_counts(long_df, value_col, school_ids=None):
    view = long_df if school_ids is None else long_df[long_df["school_id"].isin(school_ids)]
    return (
        view.drop_duplicates(["faculty_id", value_col])
        .groupby(value_col)["faculty_id"]
        .nunique()
        .rename("faculty_count")
        .reset_index()
        .sort_values(["faculty_count", value_col], ascending=[False, True])
        .reset_index(drop=True)
    )


def markdown_table(df, max_rows=None):
    view = df if max_rows is None else df.head(max_rows)
    if view.empty:
        return "_無可用資料。_"
    def clean(value):
        if isinstance(value, float):
            value = f"{value:.1f}"
        return str(value).replace("|", "\\|").replace("\n", " ")
    columns = [clean(column) for column in view.columns]
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    lines.extend("| " + " | ".join(clean(value) for value in row) + " |" for row in view.itertuples(index=False, name=None))
    return "\n".join(lines)


def department_summary():
    rows = []
    for school_id in school_order:
        faculty_count = int(faculty_df.loc[faculty_df["school_id"] == school_id, "faculty_id"].nunique())
        research_view = research_long[research_long["school_id"] == school_id]
        phd_view = phd_long[phd_long["school_id"] == school_id]
        rows.append({
            "school_id": school_id,
            "faculty_count": faculty_count,
            "research_coverage_pct": round(100 * research_view["faculty_id"].nunique() / faculty_count, 1) if faculty_count else 0.0,
            "unique_research_fields": int(research_view["research_interest_en"].nunique()),
            "phd_coverage_pct": round(100 * phd_view["faculty_id"].nunique() / faculty_count, 1) if faculty_count else 0.0,
            "unique_phd_institutions": int(phd_view["institution_en"].nunique()),
        })
    return pd.DataFrame(rows)


def shared_values(long_df, value_col, school_ids):
    view = long_df[long_df["school_id"].isin(school_ids)].drop_duplicates(["school_id", "faculty_id", value_col])
    shared = (
        view.groupby(value_col)
        .agg(department_count=("school_id", "nunique"), faculty_count=("faculty_id", "nunique"))
        .reset_index()
    )
    return shared[shared["department_count"] >= 2].sort_values(
        ["department_count", "faculty_count", value_col], ascending=[False, False, True]
    ).reset_index(drop=True)


summary = department_summary()
report = [
    "# 各校系 Research Field 與 PhD Institution 分析比較報告",
    "",
    f"產生時間：{RUN_TIMESTAMP}",
    "",
    "## 分析範圍與方法",
    "",
    f"本報告涵蓋 {len(school_order)} 個校系、{faculty_df['faculty_id'].nunique():,} 位教師。Research field 採主要研究領域與研究領域摘要分別清洗、標準化後的英文 canonical 聯集；PhD institution 同樣採預處理後的英文 canonical 值。計數單位為具有該項目的不重複教師人數。",
    "",
    "分組圖與 CSV 均保留組內觀察到的全部類別，未套用 Top-N 截斷。熱門項目表為便於閱讀只顯示前 10 或 15 名；完整數值請查閱 group comparison CSV。",
    "",
    "## 全體校系概況",
    "",
    markdown_table(summary),
    "",
    "### 全體最常見 Research Fields",
    "",
    markdown_table(top_research, max_rows=15),
    "",
    "### 全體最常見 PhD Institutions",
    "",
    markdown_table(top_phd, max_rows=15),
    "",
    "## 分組比較",
    "",
]

for group_name, school_ids in GROUPS.items():
    group_summary = summary[summary["school_id"].isin(school_ids)].set_index("school_id").loc[school_ids].reset_index()
    group_research = value_counts(research_long, "research_interest_en", school_ids)
    group_phd = value_counts(phd_long, "institution_en", school_ids)
    shared_research = shared_values(research_long, "research_interest_en", school_ids)
    shared_phd = shared_values(phd_long, "institution_en", school_ids)
    report.extend([
        f"### {group_name} 組",
        "",
        "成員：" + "、".join(f"`{school_id}`" for school_id in school_ids),
        "",
        markdown_table(group_summary),
        "",
        f"![{group_name} research fields](../analysis/descriptive-stats-after-preprocessing/group_comparison/research_fields/{group_name}.png)",
        "",
        f"![{group_name} PhD institutions](../analysis/descriptive-stats-after-preprocessing/group_comparison/phd_institutions/{group_name}.png)",
        "",
        "#### 組內熱門 Research Fields",
        "",
        markdown_table(group_research, max_rows=10),
        "",
        "#### 至少兩個校系共有的 Research Fields",
        "",
        markdown_table(shared_research, max_rows=10),
        "",
        "#### 組內熱門 PhD Institutions",
        "",
        markdown_table(group_phd, max_rows=10),
        "",
        "#### 至少兩個校系共有的 PhD Institutions",
        "",
        markdown_table(shared_phd, max_rows=10),
        "",
    ])

report.extend(["## 各校系分析", ""])
for school_id in school_order:
    school_summary_row = summary[summary["school_id"] == school_id].iloc[0]
    school_research = value_counts(research_long[research_long["school_id"] == school_id], "research_interest_en")
    school_phd = value_counts(phd_long[phd_long["school_id"] == school_id], "institution_en")
    leading_research = school_research.iloc[0]["research_interest_en"] if not school_research.empty else "無可用資料"
    leading_phd = school_phd.iloc[0]["institution_en"] if not school_phd.empty else "無可用資料"
    report.extend([
        f"### {school_id}",
        "",
        f"共 {int(school_summary_row['faculty_count'])} 位教師；research field 覆蓋率為 {school_summary_row['research_coverage_pct']:.1f}%，PhD institution 覆蓋率為 {school_summary_row['phd_coverage_pct']:.1f}%。最常見 research field 為 `{leading_research}`；最常見 PhD institution 為 `{leading_phd}`。",
        "",
        "#### Research Fields（前 10 名）",
        "",
        markdown_table(school_research, max_rows=10),
        "",
        "#### PhD Institutions（前 10 名）",
        "",
        markdown_table(school_phd, max_rows=10),
        "",
    ])

report.extend([
    "## 解讀限制",
    "",
    "- 同一位教師可有多個 research fields，因此類別人數不可直接加總為教師總數。",
    "- 主要研究領域與研究領域摘要分開保留來源、最後才合併 canonical 值；摘要中的防詐聲明等非研究文字不列入統計。",
    "- PhD institution 僅計入成功辨識且標準化的博士學歷；缺值代表來源缺漏或無法可靠解析，不代表教師沒有博士學位。",
    "- 各校系師資人數與網站資料完整度不同，絕對人數適合描述組成，不宜直接解讀為校系排名。",
    "- 熱圖呈現全部類別；大型圖檔可能較長，CSV 是精確比較與後續分析的主要依據。",
    "",
])
REPORT_OUTPUT.write_text("\n".join(report), encoding="utf-8")

expected_files = [*group_output_files, REPORT_OUTPUT]
assert len(group_output_files) == len(GROUPS) * 4
assert all(path.exists() and path.stat().st_size > 0 for path in expected_files)
assert all(len(research_group_tables[name]) == research_long[research_long["school_id"].isin(members)]["research_interest_en"].nunique() for name, members in GROUPS.items())
assert all(len(phd_group_tables[name]) == phd_long[phd_long["school_id"].isin(members)]["institution_en"].nunique() for name, members in GROUPS.items())

print("Validation passed.")
print(f"Report: {REPORT_OUTPUT}")

Validation passed.
Report: <project-root>\docs\research-field-and-phd-institution-report.md
